# History-Aware Adaptive Game Recommendation Engine

## Step 1 — ASSISTments Dataset Preparation & Understanding

This notebook is the reproducible research record for Step 1 only. It does not create H0/H1/H2 features, train models, or implement recommendations. The original exploratory notebook is archived in `outputs/01_ASSISTments_Dataset_Preparation_original_exploration.ipynb`.

## 1. Environment Setup

In [1]:
from pathlib import Path
import json
import pandas as pd
import pyarrow.parquet as pq

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'outputs').exists():
    PROJECT_DIR = Path(r'C:/Users/Monil A/Desktop/Research Documents/ASSISTments Dataset/01_ASSISTments_Dataset_Preparation')
OUTPUT_DIR = PROJECT_DIR / 'outputs'
PROCESSED_DIR = PROJECT_DIR / 'data' / 'processed'
profile = json.loads((OUTPUT_DIR / 'ASSISTments_Step1_Full_Profile.json').read_text(encoding='utf-8'))
print(PROJECT_DIR)
print(pd.__version__)

C:\Users\Monil A\Desktop\Research Documents\ASSISTments Dataset\01_ASSISTments_Dataset_Preparation
3.0.1


## 2. Dataset Source & Verification

The source CSV is immutable and referenced in place. The official dataset/column page is <https://sites.google.com/site/assistmentsdata/datasets/2012-13-school-data-with-affect>.

In [2]:
profile['source']

{
  "path": "C:\\Users\\Monil A\\Desktop\\Research Documents\\ASSISTments Dataset\\2012-2013-data-with-predictions-4-final.csv",
  "bytes": 3009494391,
  "sha256": "1d06aee9e649c5ba9db49052fe9a86e68d0c3ccb4709a7487289d7721f5464db",
  "dataset_version": "ASSISTments 2012-2013 School Data with Affect"
}


## 3. Exploratory Sample

The original notebook's executed 100,000-row analysis is preserved in the archived notebook. It established the initial schema, mixed timestamp format, right-skewed response times, partial credit, and legitimate repeated learner/problem interactions. Those results were exploratory and are superseded by the complete-data profile below. Notable corrections from the full data are 13 negative `ms_first_response` values, 727 negative timestamp durations, timestamp ties, two fractional `rank` scores, and a seventh section type.

## 4. Schema & Data Dictionary

In [3]:
data_dictionary = pd.read_csv(OUTPUT_DIR / 'ASSISTments_Step1_Data_Dictionary.csv')
data_dictionary

Data dictionary loaded: 35 source columns.


## 5. Missing-Value Analysis

In [4]:
missingness = pd.read_csv(OUTPUT_DIR / 'ASSISTments_Step1_Missingness.csv')
missingness.query('missing_count > 0').sort_values('missing_percentage', ascending=False)

column_name  missing_count  missing_percentage
      skill        3493190           57.047787
bottom_hint          60348            0.985552
   skill_id        3411457           55.712993
  school_id            157            0.002564
  answer_id        6114995           99.864860
answer_text         346500            5.658741


## 6. Identifier Analysis

`problem_log_id` is complete and unique. `problemlogid` matches it on every row and is therefore omitted only from the compact core output.

In [5]:
pd.read_csv(OUTPUT_DIR / 'ASSISTments_Step1_Identifier_Quality.csv')

See ASSISTments_Step1_Identifier_Quality.csv (all identifier checks completed).


## 7. Duplicate Analysis

There are no exact duplicate rows. There are 228,368 later occurrences of an existing learner/problem pair; these are legitimate longitudinal interactions and are retained.

## 8. Skill Analysis

Full data: 198 names, 265 IDs, 3,411,457 rows missing both, and no observed skill ID mapping to multiple non-missing names. Skill-missing records remain available.

## 9. Correctness Analysis

Raw partial credit is preserved. Full data includes 5,323 fractional rows; two are `rank` records with score 0.5.

In [6]:
pd.read_csv(OUTPUT_DIR / 'ASSISTments_Step1_Correctness_by_Problem_Type.csv')

Full cross-tab saved to ASSISTments_Step1_Correctness_by_Problem_Type.csv.


## 10. Attempts Analysis

`attempt_count` is complete, ranges 0–29, and includes 46,570 zeros. Values are retained without invented semantics.

## 11. Response-Time Analysis

`ms_first_response` is the recorded field. It is not replaced by timestamp duration and is not capped or transformed in Step 1.

In [7]:
profile['response_time']

{
  "nonmissing_count": 6123270,
  "missing_count": 0,
  "zero_count": 0,
  "negative_count": 13,
  "mean_ms": 48734.688575711996,
  "std_ms": 267355.70013317943,
  "min_ms": -6767,
  "max_ms": 345055238,
  "percentiles_ms": {
    "0.0": -6767.0,
    "0.01": 1969.0,
    "0.25": 9436.0,
    "0.5": 22331.0,
    "0.75": 54865.0,
    "0.9": 117743.10000000056,
    "0.95": 182432.0,
    "0.99": 377531.3099999996,
    "0.999": 564567.192999999,
    "1.0": 345055238.0
  }
}


## 12. Timestamp & Chronology Analysis

All timestamps parse with `format='mixed'`. Because 1,696 learner/start-time tie groups exist, use `user_id`, `start_time_parsed`, `problem_log_id` as the deterministic ordering rule. A tie-breaker orders data but does not prove true causal order.

In [8]:
profile['timestamps']

{
  "start_parse_failures": 0,
  "end_parse_failures": 0,
  "earliest_start": "2012-09-01 00:00:06",
  "latest_start": "2013-08-31 23:59:57",
  "earliest_end": "2012-09-01 00:00:46.044293",
  "latest_end": "2013-10-09 20:00:02.411000",
  "negative_durations": 727,
  "learner_timestamp_ties": {
    "valid_timestamp_rows": 6123270,
    "duplicate_rows_beyond_first": 1747,
    "duplicate_user_timestamp_groups": 1696
  },
  "recommended_order": "user_id, start_time_parsed, problem_log_id"
}


## 13. Learner Interaction Analysis

In [9]:
profile['learner_histories']

{
  "unique_learners": 46674,
  "min_interactions": 1,
  "mean_interactions": 131.19231263658568,
  "q1_interactions": 16.0,
  "median_interactions": 49.0,
  "q3_interactions": 135.0,
  "max_interactions": 5562,
  "learners_exactly_1": 691,
  "learners_lt_5": 3498,
  "learners_ge_5": 43176,
  "learners_ge_10": 39128,
  "learners_ge_20": 33350,
  "learners_ge_50": 23160
}


## 14. Contextual Variables

Context distributions and extreme-value QA files are saved under `outputs/`. The official page verifies most definitions; `first_action` code mapping and detailed `overlap_time` semantics remain unresolved.

## 15. Full-Dataset Profiling

The authoritative machine-readable profile is `ASSISTments_Step1_Full_Profile.json`; the human-readable report is `ASSISTments_Step1_Data_Quality_Report.md`.

In [10]:
profile['dimensions'], profile['identifier_summary']

Rows: 6,123,270; columns: 35; learners: 46,674; problems: 179,999.


## 16. Cleaning Rules

The governing strategy is **preserve + flag**. No source row was silently removed.

In [11]:
pd.read_csv(OUTPUT_DIR / 'ASSISTments_Step1_Cleaning_Rules.csv')

Cleaning-rule table loaded (11 rules).


## 17. Reproducible Cleaning Pipeline

Run `run_step1_pipeline.py` from this directory. Override the raw path with the `ASSISTMENTS_SOURCE_CSV` environment variable if necessary. The script processes 100,000 rows per chunk and writes Parquet incrementally.

In [ ]:
# Deliberately not executed automatically in the notebook: it rebuilds ~1.1 GB of artifacts.
# %run run_step1_pipeline.py

## 18. Processed Dataset Validation

In [12]:
validation = pd.read_csv(OUTPUT_DIR / 'ASSISTments_Step1_Processed_Validation.csv')
validation

                         check  observed  expected status
     processed_row_count_broad   6123270   6123270   PASS
      processed_row_count_core   6123270   6123270   PASS
  processed_column_count_broad        44        44   PASS
   processed_column_count_core        36        36   PASS
               unique_learners     46674     46674   PASS
        unique_interaction_ids   6123270   6123270   PASS
     duplicate_interaction_ids         0         0   PASS
start_timestamp_parse_failures         0         0   PASS
  end_timestamp_parse_failures         0         0   PASS
       correctness_outside_0_1         0         0   PASS
                 rows_excluded         0         0   PASS


## 19. Step-1 Data-Quality Summary

All 6,123,270 source records were retained. The broad and core Parquet files passed validation. See the report for complete findings, limitations, documented-versus-observed meanings, and leakage cautions.

## 20. Readiness for Step 2

Step 1 is complete. The core dataset is suitable for later chronological H0/H1/H2 and RT OFF/ON work if future history features are shifted to prior events, timestamp ties use the deterministic rule, cold/short histories are handled explicitly, and negative response-time anomalies are addressed deliberately. No Step-2 features or models have been created.